# SEER Breast Cancer 2010–2017 — Modeling (03)

Fits every model of the factorial benchmark, following `Documents/Methods_Modeling.md` and spec v4. All fitting code lives in `seer_modeling.py`; this notebook only runs it.

**In each of the 15 splits (3 repeats × 5 folds):**
1. Fit the encoder and the censoring model Ĝ(t | X) on outer-train, then compute IPCW weights for τ = 36, 60 and 120 months.
2. Fit the factorial grid on inner-train:
   - 6 cells (Zero / Discard / IPCW × none / SMOTE-NC 1:1);
   - 3 classifiers (logistic regression, random forest, XGBoost);
   - 3 horizons.

   That is 54 fits per split.
3. Fit the two reference models on inner-train: cause-specific Cox and cause-specific RSF.
4. Save, for every inner-validation and test patient, the IPCW weights and all predictions. Nothing is evaluated here.

**Outputs** go to `outputs_modeling/`:
- `split_r{r}_k{k}.parquet`: weights and predictions;
- `split_r{r}_k{k}.json`: diagnostics, i.e. weight checks, training-set sizes, imbalance ratios before and after SMOTE, and fit times.

Finished splits are skipped, so an interrupted run simply resumes. Threshold selection and all metrics are done in `04_evaluation.ipynb`.

**Run mode:**
- `"pilot"` fits split (0, 0) only, to measure runtime.
- `"full"` fits all 15 splits. Run it in the background, e.g. `jupyter nbconvert --execute`.

In [ ]:
from pathlib import Path
import config
import json
import time
import warnings

import pandas as pd

import seer_modeling as sm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RUN_MODE = "full"             # "pilot" = split (0, 0) only; "full" = all 15 splits

NOTEBOOK_DIR = config.WORK_DIR    # all outputs below this folder (config.py)
PREP_DIR = NOTEBOOK_DIR / "outputs_model_prep"
OUT_DIR = NOTEBOOK_DIR / "outputs_modeling"

data = sm.load_prep(PREP_DIR)   # verifies every checksum in prep_manifest.json
SPLITS = [(0, 0)] if RUN_MODE == "pilot" else [
    (r, k) for r in range(data.manifest["cv"]["n_repeats"]) for k in range(data.manifest["cv"]["n_folds"])
]
print(f"Mode: {RUN_MODE} | splits: {len(SPLITS)} | patients: {len(data.cohort):,} | predictors: {len(data.x_cols)}")

## Fit all splits

In [ ]:
t_start = time.time()
for r, k in SPLITS:
    print(f"--- split ({r}, {k}) ---", flush=True)
    sm.run_split(data, r, k, OUT_DIR, log=lambda msg: print(msg, flush=True))
print(f"\nFinished {len(SPLITS)} split(s) in {(time.time() - t_start) / 60:.1f} min")

## Diagnostics of the fitted splits

**Weights:** the mean untruncated weight must be about 1 (checked during fitting). The table also shows the 99th-percentile cap and the Kish effective sample size, which feed Table S2.

**Cells:** training rows, imbalance ratio before and after SMOTE-NC, and the number of synthetic patients, which feed Table T2.

In [ ]:
diags = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(OUT_DIR.glob("split_r*_k*.json"))]

weights = pd.DataFrame([
    {"split": tuple(d["split"]), "tau": int(tau), **{k: v for k, v in w.items() if k != "quantiles_determinable"}}
    for d in diags for tau, w in d["weights"].items()
])
display(weights.groupby("tau")[["mean_untruncated_all", "cap_p99", "max_untruncated", "share_truncated",
                                "n_determinable", "kish_ess_truncated"]].mean().round(3))

cells = pd.DataFrame([
    {"split": tuple(d["split"]), "column": col, **info} for d in diags for col, info in d["cells"].items()
])
cells[["model", "cell", "tau"]] = cells["column"].str.split("|", expand=True)
summary = (
    cells[cells["model"] == "lr"]
    .groupby(["tau", "cell"], sort=False)[["n_rows", "n_pos", "ir_before", "ir_after", "n_synthetic", "kish_ess"]]
    .mean().round(2)
)
display(summary)

In [ ]:
timing = cells.groupby(["model"])["fit_seconds"].agg(["mean", "max", "sum"]).round(1)
display(timing)
refs = pd.DataFrame([{"split": tuple(d["split"]), "reference": ref, **v}
                     for d in diags for ref, v in d["references"].items()])
display(refs.groupby("reference")["fit_predict_seconds"].mean().round(1))
per_split = pd.Series({tuple(d["split"]): d["total_seconds"] / 60 for d in diags}, name="minutes").round(1)
display(per_split.to_frame())
print(f"Projected time for all 15 splits: {per_split.mean() * 15 / 60:.1f} h")